In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashwatwork/eeg-psychiatric-disorders-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.72M/4.72M [00:00<00:00, 37.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shashwatwork/eeg-psychiatric-disorders-dataset/versions/1


In [3]:
base = pd.read_csv('/kaggle/input/eeg-psychiatric-disorders-dataset/EEG.machinelearing_data_BRMH.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/eeg-psychiatric-disorders-dataset/EEG.machinelearing_data_BRMH.csv'

# **Prep. de datos**


In [ ]:
selected_disorders = [
    "Alcohol use disorder",
    "Major depressive disorder",
    "Schizophrenia"
]

df = base[base["specific.disorder"].isin(selected_disorders)]
df.head()

In [ ]:
X = df.drop(columns=[
    "no.",
    "eeg.date",
    "main.disorder",
    "specific.disorder"
])
X = X.drop(columns=["Unnamed: 122"])
y = df["specific.disorder"]

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns


X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())


for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# **Regresion logistica**

In [ ]:
model = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1500
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

print("F1 macro:", f1_score(y_test, y_pred, average='macro'))
print(confusion_matrix(y_test, y_pred))

# **K-Means**

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=1)
kmeans.fit(X_train)

# Predecir SOLO sobre test
clusters = kmeans.predict(X_test)

# Verificar tamaños
print(len(y_test), len(clusters))

In [ ]:
cluster_map = {
    0: "Schizophrenia",
    1: "Alcohol use disorder"
}

mapped_clusters = pd.Series(clusters).map(cluster_map)



accuracy = accuracy_score(y_test.values, mapped_clusters)
print("Accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Si X_test es DataFrame lo convertimos a array
X_plot = X_test.values if hasattr(X_test, "values") else X_test

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

scatter = ax.scatter(
    X_plot[:, 0],
    X_plot[:, 1],
    X_plot[:, 2],
    c=clusters  # coloreado automático por cluster
)

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_zlabel("Feature 3")
ax.set_title("Clusters predichos por KMeans (3D)")

plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_numeric = le.fit_transform(y_test)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(
    X_plot[:, 0],
    X_plot[:, 1],
    X_plot[:, 2],
    c=y_numeric
)

ax.set_title("Clases reales (3D)")
plt.show()

# **SVM**

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for C in [0.5, 1, 2, 4, 8, 16]:
    svm = SVC(kernel='rbf', C=C, gamma='scale')
    scores = cross_val_score(svm, X_train_scaled, y_train, cv=5)
    print(f"C={C} -> CV mean: {np.mean(scores):.3f}")

In [ ]:
### ESTE MODELO SOBRE AJUSTA EL TRAIN
svm_model = SVC(kernel='rbf', C=8.0, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

y_pred = svm_model.predict(X_test_scaled)


accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
train_pred = svm_model.predict(X_train_scaled)
print("Train accuracy:", accuracy_score(y_train, train_pred))

# **KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier


for k in [1, 3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(f"k={k} -> CV mean: {np.mean(scores):.3f}")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

In [ ]:
print("Test accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# **PCA + SVM**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_test_pca)

In [ ]:
print("Test accuracy:", accuracy_score(y_test, y_pred))

train_pred = svm_model.predict(X_train_pca)
print("Train accuracy:", accuracy_score(y_train, train_pred))

# **Conclusiones rápidas**

| Modelo                  | Test Accuracy |
|--------------------------|---------------|
| KMeans (k=2)            | 0.547         |
| KNN (k=5)               | 0.542         |
| PCA + SVM (C=1)         | 0.571         |
| Regresión Logística     | 0.619         |
| SVM (C=1)               | 0.601         |
| SVM (C=8)(sobre ajuste)              | 0.714         |

Los modelos supervisados superan al clustering no supervisado, lo que confirma la existencia de señal discriminativa en las variables. Sin embargo, el rendimiento promedio en validación cruzada se mantiene cercano al 60%, lo que indica un solapamiento significativo entre clases. El uso de modelos más complejos incrementa el sobreajuste sin mejorar la capacidad de generalización.
Se buscará ajustar las señales para tener mayor información.

De momento el mejor modelo es Regresión Logística o SVM con C=1
Porque son:
* Más estables
* Menos propensos a sobreajuste
* Consistentes con validación cruzada